# 04 — CUDA YOLOv8n WBC training

**Purpose:** Run one CUDA smoke epoch, verify checkpoints, then run or resume full transfer learning on Windows.

**Required inputs:** Valid processed data, `Python (leukemia-yolo)` kernel, CUDA-enabled PyTorch and `yolov8n.pt` download/cache access.

**Saved outputs:** Timestamped runs in `outputs/checkpoints/` with best.pt, last.pt, results.csv, results.png, logs and effective config.

## 1. Load the central configuration

The root is found from the notebook location or the installed package. Set `LEUKEMIA_YOLO_CONFIG` only for synthetic smoke tests or an alternate config.

In [ ]:
from pathlib import Path
import os, sys
search = [Path.cwd(), *Path.cwd().parents]
if os.environ.get("LEUKEMIA_YOLO_ROOT"):
    search.insert(0, Path(os.environ["LEUKEMIA_YOLO_ROOT"]))
for candidate in search:
    if (candidate / "config.example.yaml").is_file() and (candidate / "src" / "config.py").is_file():
        sys.path.insert(0, str(candidate))
        break
from src.config import load_config, ensure_outputs
cfg = load_config()
out = ensure_outputs(cfg)
os.chdir(cfg["root"])
print("Project root:", cfg["root"])
print("Configuration:", cfg["config_path"])

## A. Check GPU and dataset readiness

CUDA failure stops this notebook. System RAM and dedicated NVIDIA VRAM are separate resources.

In [ ]:
from src.training import training_readiness, stream_training, checkpoint_for_run, results_table, progress_columns
from src.environment import nvidia_smi
from IPython.display import display, Image as IPImage
import pandas as pd
ready = training_readiness(cfg)
display(pd.DataFrame([{"setting": k, "value": str(v)} for k,v in ready.items()]))
print("nvidia-smi:", nvidia_smi())

## B. Load pretrained detection checkpoint

This must be the YOLOv8n **detection** checkpoint. First use may download it from Ultralytics.

In [ ]:
from ultralytics import YOLO
model = YOLO("yolov8n.pt")
assert model.task == "detect", f"Unexpected task: {model.task}"
print("PASS — pretrained object detector loaded:", cfg["model"])

## C. Show effective plan

Default batch 0.75 asks Ultralytics AutoBatch to target a fraction of GPU memory. Automatic disk caching requires Windows to identify an SSD and enough free space; unknown media type disables cache.

In [ ]:
from src.training import choose_cache
plan = {"epochs": cfg["epochs"], "imgsz": cfg["imgsz"], "batch": cfg["batch"],
        "device": cfg["device"], "amp": cfg["amp"], "workers": cfg["workers"],
        "patience": cfg["patience"], "seed": cfg["seed"], "plots": True,
        "save": True, "save_period": 5, "cache": choose_cache(cfg)}
display(pd.DataFrame(plan.items(), columns=["setting", "value"]))
print("If Windows DataLoader stalls, set workers: 0 in config.local.yaml and start a new run.")

## D. Run one smoke epoch

Training runs in a Windows-safe child process through `sys.executable`; its stdout and stderr stream here and to a log. Set `RESUME_LAST` to an unfinished last.pt to skip a new smoke run.

In [ ]:
RESUME_LAST = None  # Optional Path(".../weights/last.pt") from an interrupted main run.
ALLOW_BATCH_FALLBACK = True  # Logged AutoBatch/OOM retry at batch=4 in a NEW run; failed run is preserved.
smoke_run = None
if RESUME_LAST is None:
    smoke_run = stream_training(cfg, kind="smoke", allow_batch_fallback=ALLOW_BATCH_FALLBACK)
    print("Smoke run:", smoke_run)
else:
    RESUME_LAST = Path(RESUME_LAST).expanduser().resolve()
    print("Resuming existing main run from", RESUME_LAST)

## E. Verify smoke checkpoint

A completed one-epoch CUDA run, loadable last.pt and results.csv are required before main training.

In [ ]:
if smoke_run:
    smoke_last = checkpoint_for_run(smoke_run, "last.pt")
    smoke_model = YOLO(str(smoke_last))
    assert smoke_model.task == "detect" and len(results_table(smoke_run)) >= 1
    print("PASS — CUDA smoke training and checkpoint load succeeded:", smoke_last)
else:
    assert RESUME_LAST.is_file()
    print("PASS — existing last.pt selected for resume")

## F. Train or resume main experiment

The main run requires the completed smoke run. Stopping leaves last.pt; resume by setting `RESUME_LAST` above. Reruns use new timestamped directories.

In [ ]:
main_run = stream_training(cfg, kind="main", smoke_run=smoke_run, resume=RESUME_LAST,
                           allow_batch_fallback=ALLOW_BATCH_FALLBACK)
print("Main run:", main_run)

## G. Review saved training outputs

A completed run must contain both checkpoints, CSV, plot and metadata.

In [ ]:
from src.config import read_json
metadata = read_json(main_run / "run_metadata.json")
for name in ("weights/best.pt", "weights/last.pt", "results.csv", "results.png", "effective_training_config.json"):
    assert (main_run / name).is_file(), name
display(metadata)
display(IPImage(filename=str(main_run / "results.png")))

## H. GPU utilization and peak VRAM

These values describe this run; 100% utilization is neither required nor promised.

In [ ]:
display(pd.DataFrame([{"GPU": metadata["gpu_start"]["gpu_name"],
                         "peak allocated GiB": metadata.get("gpu_peak_allocated_gib"),
                         "actual batch": metadata.get("actual_batch"),
                         "nvidia-smi at end": metadata.get("nvidia_smi_end")}]))

## I. Latest epoch progress

The CSV reader tolerates a partially written last row during a running experiment.

In [ ]:
progress = progress_columns(results_table(main_run))
display(progress.tail(10))
assert not progress.empty and metadata["completed_epochs"] > 0

## J. Load best checkpoint and finish

This checkpoint is used for later held-out evaluation and inference.

In [ ]:
best_path = checkpoint_for_run(main_run)
best_model = YOLO(str(best_path))
assert best_model.task == "detect"
print("PASS — completed CUDA training. Best checkpoint:", best_path)
print("Validation metrics are in results.csv; held-out test metrics are produced only by notebook 05.")

## Next notebook

Open **05_Model_Evaluation.ipynb**. Never tune hyperparameters from its held-out test results.